# 1. Tujuan

Notebook ini melakukan training model klasifikasi final untuk kondisi keuangan bulanan personal user menggunakan dataset labeling baru.

## Dataset dan target
- Dataset: FINARY_MONTHLY_CONDITION_DATASET.csv
- Target: monthly_financial_condition
- Label mapping:
  - 0 = survival
  - 1 = stable
  - 2 = growth

Fokus notebook ini hanya training, evaluasi, dan ekspor artifact final (tanpa integrasi FastAPI pada tahap ini).

## 2. Imports

Import library utama untuk preprocessing, TensorFlow custom training loop, evaluasi, dan export artifact.

In [3]:
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 3. Konfigurasi

Konfigurasi training final dan path artifact.

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("FINARY_MONTHLY_CONDITION_DATASET.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "monthly_financial_condition"
LABEL_MAPPING = {"0": "survival", "1": "stable", "2": "growth"}
EXCLUDE_FEATURE_COLS = {
    "monthly_financial_condition",
    "monthly_financial_condition_label",
    "financial_scenario",
}

EPOCHS = 150
BATCH_SIZE = 128
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CLIP_NORM = 1.0
MIN_LR = 1e-5
LR_PATIENCE = 6
LR_FACTOR = 0.5

print("DATA_PATH:", DATA_PATH)
print("TARGET:", TARGET)

DATA_PATH: FINARY_MONTHLY_CONDITION_DATASET.csv
TARGET: monthly_financial_condition


## 4. Load Dataset

Load dataset hasil labeling final untuk training klasifikasi.

In [5]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), "Cols:", df.shape[1])
display(df.head(3))

Rows: 3000 Cols: 45


,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,...,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium,monthly_financial_condition,monthly_financial_condition_label
0,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,3,...,True,False,False,False,False,True,True,False,0,survival
1,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,4,...,True,False,False,False,False,True,True,False,0,survival
2,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,5,...,False,False,False,False,False,True,True,False,0,survival


## 5. Minimal Dataset Validation

Validasi ringkas: target tersedia, value target valid, dan distribusi kelas awal.

In [6]:
if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found in dataset")

unique_targets = sorted(df[TARGET].dropna().astype(int).unique().tolist())
if unique_targets != [0, 1, 2]:
    raise ValueError(f"Invalid target values. Expected [0,1,2], got {unique_targets}")

if df[TARGET].isna().any():
    raise ValueError("Target contains missing values")

dist = df[TARGET].value_counts().sort_index()
dist_pct = (dist / dist.sum() * 100).round(2)
print("Target distribution (count):")
print(dist)
print("Target distribution (%):")
print(dist_pct)

Target distribution (count):
monthly_financial_condition
0     651
1     438
2    1911
Name: count, dtype: int64
Target distribution (%):
monthly_financial_condition
0    21.7
1    14.6
2    63.7
Name: count, dtype: float64


## 6. Feature/Target Preparation

Siapkan fitur numerik/boolean valid dan target tanpa leakage.

In [7]:
df_prep = df.copy()

# Convert boolean columns to integers.
for col in df_prep.columns:
    if df_prep[col].dtype == bool:
        df_prep[col] = df_prep[col].astype(int)

# Define candidate feature columns by exclusion rule.
candidate_cols = [c for c in df_prep.columns if c not in EXCLUDE_FEATURE_COLS]

# Keep only numeric columns as model input.
X_df = df_prep[candidate_cols].select_dtypes(include=["number"]).copy()
y = df_prep[TARGET].astype(int)

if TARGET in X_df.columns:
    raise ValueError("Target leakage detected: target found in features")

if "monthly_financial_condition_label" in X_df.columns:
    raise ValueError("String label column leaked into features")

# Fill numeric missing values with median.
X_df = X_df.fillna(X_df.median(numeric_only=True))
feature_columns = X_df.columns.tolist()

print("Feature count:", len(feature_columns))
print("Sample features:", feature_columns[:20])
display(X_df.head(3))

Feature count: 42
Sample features: ['monthly_income', 'monthly_expense_total', 'savings_rate', 'budget_goal', 'credit_score', 'debt_to_income_ratio', 'loan_payment', 'investment_amount', 'subscription_services', 'emergency_fund', 'transaction_count', 'fraud_flag', 'discretionary_spending', 'essential_spending', 'rent_or_mortgage', 'financial_advice_score', 'actual_savings', 'savings_goal_met', 'net_cash_flow', 'expense_ratio']


,monthly_income,monthly_expense_total,savings_rate,budget_goal,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,emergency_fund,...,category_Healthcare,category_Insurance,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium
0,3119.58,3212.07,0.38,3676.11,721.0,0.56,125.77,689.22,3,510.58,...,0,0,1,0,0,0,0,1,1,0
1,3262.44,3732.81,0.10,2607.17,670.0,0.42,454.19,360.34,4,1154.41,...,0,0,1,0,0,0,0,1,1,0
2,2931.20,3335.58,0.15,3004.14,691.0,0.24,971.82,0.00,5,1433.02,...,1,0,0,0,0,0,0,1,1,0


## 7. Train/Validation/Test Split

Gunakan stratified split 70/15/15 dengan random_state=42.

In [8]:
X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_df,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

def label_dist(series: pd.Series) -> pd.DataFrame:
    vc = series.value_counts().sort_index()
    pct = (vc / vc.sum() * 100).round(2)
    return pd.DataFrame({"count": vc, "pct": pct})

print("Shapes:")
print("Train:", X_train_df.shape, y_train.shape)
print("Val  :", X_val_df.shape, y_val.shape)
print("Test :", X_test_df.shape, y_test.shape)

print("\nTrain distribution:")
display(label_dist(y_train))
print("Val distribution:")
display(label_dist(y_val))
print("Test distribution:")
display(label_dist(y_test))

Shapes:
Train: (2100, 42) (2100,)
Val  : (450, 42) (450,)
Test : (450, 42) (450,)

Train distribution:


,count,pct
monthly_financial_condition,,
0,456,21.71
1,306,14.57
2,1338,63.71


Val distribution:


,count,pct
monthly_financial_condition,,
0,97,21.56
1,66,14.67
2,287,63.78


Test distribution:


,count,pct
monthly_financial_condition,,
0,98,21.78
1,66,14.67
2,286,63.56


## 8. Feature Selection

Seleksi fitur dilakukan hanya dari train split (tanpa leakage), lalu hasilnya diterapkan konsisten ke train/val/test, artifact, dan inference.

In [9]:
# Automatic feature selection using train split only (to avoid leakage)

CORR_THRESHOLD = 0.98
MIN_FEATURES = 18
KEEP_RATIO = 0.65

feature_selection_report = {}

# Start from train features
X_train_fs = X_train_df.copy()

# 1) Remove constant features
constant_cols = [c for c in X_train_fs.columns if X_train_fs[c].nunique(dropna=False) <= 1]
if constant_cols:
    X_train_fs = X_train_fs.drop(columns=constant_cols)

# 2) Mutual information on current train features
mi_scores = mutual_info_classif(
    X_train_fs.values,
    y_train.values,
    random_state=SEED,
)
mi_series = pd.Series(mi_scores, index=X_train_fs.columns)

# 3) Correlation pruning with MI tie-breaker
corr = X_train_fs.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

corr_drop = set()
for col in upper.columns:
    high_corr_features = upper.index[upper[col] > CORR_THRESHOLD].tolist()
    for row_feature in high_corr_features:
        if row_feature in corr_drop or col in corr_drop:
            continue
        # Keep feature with higher MI score
        drop_col = row_feature if mi_series[row_feature] < mi_series[col] else col
        corr_drop.add(drop_col)

if corr_drop:
    X_train_fs = X_train_fs.drop(columns=sorted(corr_drop))

# Recompute MI after correlation pruning
mi_scores_post = mutual_info_classif(
    X_train_fs.values,
    y_train.values,
    random_state=SEED,
)
mi_post = pd.Series(mi_scores_post, index=X_train_fs.columns)

# 4) Tree-based importance
et = ExtraTreesClassifier(
    n_estimators=400,
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1,
)
et.fit(X_train_fs, y_train.values)
imp_post = pd.Series(et.feature_importances_, index=X_train_fs.columns)

# 5) Combined ranking score (lower rank sum = better)
mi_rank = mi_post.rank(ascending=False, method="average")
imp_rank = imp_post.rank(ascending=False, method="average")
combined_rank = (mi_rank + imp_rank).sort_values(ascending=True)

k = int(max(MIN_FEATURES, round(len(combined_rank) * KEEP_RATIO)))
k = min(k, len(combined_rank))
selected_feature_columns = combined_rank.index[:k].tolist()

# Apply selected features consistently
X_train_df = X_train_df[selected_feature_columns].copy()
X_val_df = X_val_df[selected_feature_columns].copy()
X_test_df = X_test_df[selected_feature_columns].copy()
X_df = X_df[selected_feature_columns].copy()

feature_columns = selected_feature_columns.copy()

feature_selection_report = {
    "initial_feature_count": int(len(candidate_cols)),
    "after_numeric_filter_count": int(len([c for c in candidate_cols if c in df_prep[candidate_cols].select_dtypes(include=['number']).columns])),
    "constant_dropped_count": int(len(constant_cols)),
    "constant_dropped": constant_cols,
    "corr_threshold": CORR_THRESHOLD,
    "corr_dropped_count": int(len(corr_drop)),
    "corr_dropped": sorted(list(corr_drop)),
    "selected_count": int(len(selected_feature_columns)),
    "selected_feature_columns": selected_feature_columns,
    "selection_method": "MI + ExtraTrees rank aggregation on train split",
}

print("Feature selection summary:")
print("- Initial numeric feature count:", len(df_prep[candidate_cols].select_dtypes(include=['number']).columns))
print("- Dropped constant:", len(constant_cols))
print("- Dropped high correlation:", len(corr_drop))
print("- Selected feature count:", len(selected_feature_columns))
print("- Top selected features:", selected_feature_columns[:20])

Feature selection summary:
- Initial numeric feature count: 42
- Dropped constant: 0
- Dropped high correlation: 0
- Selected feature count: 27
- Top selected features: ['saving_behavior', 'expense_ratio', 'actual_savings', 'net_cash_flow', 'monthly_income', 'monthly_expense_total', 'lifestyle_burden', 'savings_goal_met', 'spending_efficiency', 'financial_buffer', 'discretionary_spending', 'budget_goal', 'savings_rate', 'rent_or_mortgage', 'debt_pressure', 'emergency_fund', 'category_Education', 'financial_advice_score', 'credit_score', 'category_Entertainment']


## 8. Scaling

Gunakan RobustScaler, fit hanya pada train set, lalu transform val/test.

In [10]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_df.values.astype("float32"))
X_val_scaled = scaler.transform(X_val_df.values.astype("float32"))
X_test_scaled = scaler.transform(X_test_df.values.astype("float32"))

input_dim = X_train_scaled.shape[1]
print("input_dim:", input_dim)

input_dim: 27


## 9. Class Imbalance Handling

Hitung class weight otomatis dari train set dan perkuat minoritas secara ringan.

In [11]:
classes = np.array([0, 1, 2], dtype=int)
base_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.values)
class_weight = {int(c): float(w) for c, w in zip(classes, base_weights)}

# Slight boost for minority classes to avoid collapse to growth.
class_weight[0] *= 1.10
class_weight[1] *= 1.15

# Normalize weights around mean=1 for numerical stability.
w_mean = np.mean(list(class_weight.values()))
class_weight = {k: float(v / w_mean) for k, v in class_weight.items()}

print("Class weights:", class_weight)

def to_onehot(y_arr: np.ndarray, num_classes: int = 3) -> np.ndarray:
    y_arr = y_arr.astype(int)
    out = np.zeros((len(y_arr), num_classes), dtype=np.float32)
    out[np.arange(len(y_arr)), y_arr] = 1.0
    return out

y_train_oh = to_onehot(y_train.values, 3)
y_val_oh = to_onehot(y_val.values, 3)
y_test_oh = to_onehot(y_test.values, 3)

Class weights: {0: 1.0461137513414076, 1: 1.6297743648349197, 2: 0.3241118838236729}


## 10. Build Final TensorFlow Model

Bangun Residual MLP (Functional API) untuk klasifikasi 3 kelas.

In [12]:
@tf.keras.utils.register_keras_serializable(package="finary")
class ResidualDenseBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        units: int,
        dropout: float,
        l2: float,
        activation: str = "gelu",
        name: str | None = None,
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        reg = tf.keras.regularizers.l2(l2)
        self.units = units
        self.dropout = dropout
        self.l2 = l2
        self.activation = activation

        self.dense1 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.drop1 = tf.keras.layers.Dropout(dropout)

        self.dense2 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.drop2 = tf.keras.layers.Dropout(dropout)

        self.proj = None

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "units": self.units,
                "dropout": self.dropout,
                "l2": self.l2,
                "activation": self.activation,
            }
        )
        return config

    def _act(self, x):
        return tf.keras.activations.gelu(x) if self.activation.lower() == "gelu" else tf.nn.relu(x)

    def build(self, input_shape):
        in_units = int(input_shape[-1])
        if in_units != self.units:
            self.proj = tf.keras.layers.Dense(self.units)
        super().build(input_shape)

    def call(self, x, training=False):
        skip = self.proj(x) if self.proj is not None else x

        y = self.dense1(x)
        y = self.bn1(y, training=training)
        y = self._act(y)
        y = self.drop1(y, training=training)

        y = self.dense2(y)
        y = self.bn2(y, training=training)
        y = self._act(y)
        y = self.drop2(y, training=training)

        return skip + y


def build_residual_mlp(input_dim: int) -> tf.keras.Model:
    reg = tf.keras.regularizers.l2(WEIGHT_DECAY)
    inp = tf.keras.Input(shape=(input_dim,), name="features")

    x = tf.keras.layers.Dense(256, kernel_regularizer=reg, name="proj_dense")(inp)
    x = tf.keras.layers.BatchNormalization(name="proj_bn")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="proj_act")(x)
    x = tf.keras.layers.Dropout(0.25, name="proj_dropout")(x)

    x = ResidualDenseBlock(256, dropout=0.25, l2=WEIGHT_DECAY, activation="gelu", name="res_block_1")(x)
    x = ResidualDenseBlock(128, dropout=0.20, l2=WEIGHT_DECAY, activation="gelu", name="res_block_2")(x)
    x = ResidualDenseBlock(64, dropout=0.15, l2=WEIGHT_DECAY, activation="gelu", name="res_block_3")(x)

    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dense(64, kernel_regularizer=reg, name="head_dense")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="head_act")(x)
    x = tf.keras.layers.Dropout(0.15, name="head_dropout")(x)

    out = tf.keras.layers.Dense(3, activation="softmax", name="class_output")(x)
    return tf.keras.Model(inp, out, name="finary_classification_residual_mlp")


model = build_residual_mlp(input_dim)
model.summary()

Model: "finary_classification_residual_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 27)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dense (Dense)              │ (None, 256)            │         7,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_bn (BatchNormalization)    │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_act (Activation)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dropout (Dropout)          │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_1                     │ (None, 256)            │       133,632 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_2                     │ (None, 128)            │        83,328 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_3                     │ (None, 64)             │        21,184 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_bn (BatchNormalization)    │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense (Dense)              │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_act (Activation)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_output (Dense)            │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 250,947 (980.26 KB)

 Trainable params: 248,515 (970.76 KB)

 Non-trainable params: 2,432 (9.50 KB)

## 11. Custom Component

Gunakan custom weighted categorical crossentropy untuk memasukkan class weight ke training loop.

In [13]:
_ce = tf.keras.losses.CategoricalCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)


def weighted_categorical_crossentropy(class_weight_dict: dict[int, float]):
    w = np.array([class_weight_dict[i] for i in [0, 1, 2]], dtype=np.float32)
    w = w * (3.0 / float(w.sum()))
    w_t = tf.constant(w, dtype=tf.float32)

    def loss(y_true, y_pred):
        ce = _ce(y_true, y_pred)
        sample_w = tf.reduce_sum(y_true * w_t, axis=-1)
        return sample_w * ce

    return loss


loss_fn = weighted_categorical_crossentropy(class_weight)
print("Custom loss ready: weighted categorical crossentropy")

Custom loss ready: weighted categorical crossentropy


## 12. Custom GradientTape Training Loop

Siapkan dataset pipeline, optimizer, train_step, dan val_step tanpa model.fit.

In [14]:
def make_tf_dataset(X: np.ndarray, y_onehot: np.ndarray, batch_size: int, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y_onehot.astype(np.float32)))
    if training:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def macro_f1_from_probs(y_true: np.ndarray, probs: np.ndarray) -> float:
    y_pred = np.argmax(probs, axis=1)
    return float(f1_score(y_true, y_pred, average="macro"))


train_ds = make_tf_dataset(X_train_scaled, y_train_oh, BATCH_SIZE, training=True)
val_ds = make_tf_dataset(X_val_scaled, y_val_oh, BATCH_SIZE, training=False)
test_ds = make_tf_dataset(X_test_scaled, y_test_oh, BATCH_SIZE, training=False)

optimizer_cls = getattr(tf.keras.optimizers, "AdamW", None)
if optimizer_cls is not None:
    optimizer = optimizer_cls(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, clipnorm=CLIP_NORM)
    optimizer_name = "AdamW"
else:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIP_NORM)
    optimizer_name = "Adam"

train_acc_m = tf.keras.metrics.CategoricalAccuracy(name="train_accuracy")
val_acc_m = tf.keras.metrics.CategoricalAccuracy(name="val_accuracy")


@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        probs = model(xb, training=True)
        loss_vals = loss_fn(yb, probs)
        loss = tf.reduce_mean(loss_vals)
        if model.losses:
            loss += tf.add_n(model.losses)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_acc_m.update_state(yb, probs)
    return loss


@tf.function
def val_step(xb, yb):
    probs = model(xb, training=False)
    loss_vals = loss_fn(yb, probs)
    loss = tf.reduce_mean(loss_vals)
    if model.losses:
        loss += tf.add_n(model.losses)
    val_acc_m.update_state(yb, probs)
    return loss, probs

print("Optimizer:", optimizer_name)

Optimizer: AdamW


## 13. Training Per-Epoch (Tanpa TensorBoard)

Training manual per-epoch dengan `tf.GradientTape` + early stopping + restore best weights (tanpa logging TensorBoard).

In [15]:
best_val_macro_f1 = -1.0
best_val_loss = np.inf
best_epoch = 0
best_weights = None
bad_epochs = 0
lr_bad_epochs = 0

history = []

for epoch in range(1, EPOCHS + 1):
    train_acc_m.reset_state()
    val_acc_m.reset_state()

    # ===================== TRAIN =====================
    train_losses = []
    for xb, yb in train_ds:
        loss = train_step(xb, yb)
        train_losses.append(float(loss.numpy()))

    # ===================== VALIDATION =====================
    val_losses = []
    val_probs_all = []
    val_y_all = []

    for xb, yb in val_ds:
        vloss, probs = val_step(xb, yb)
        val_losses.append(float(vloss.numpy()))
        val_probs_all.append(probs.numpy())
        val_y_all.append(np.argmax(yb.numpy(), axis=1))

    val_probs_all = np.concatenate(val_probs_all, axis=0)
    val_y_all = np.concatenate(val_y_all, axis=0)

    # ===================== METRICS =====================
    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    train_acc = float(train_acc_m.result().numpy())
    val_acc = float(val_acc_m.result().numpy())
    val_macro_f1 = macro_f1_from_probs(val_y_all, val_probs_all)
    lr = float(optimizer.learning_rate.numpy())

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
            "learning_rate": lr,
        }
    )

    # ===================== PRINT =====================
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f} | "
        f"lr={lr:.6f}"
    )

    # ===================== EARLY STOPPING & LR =====================
    improved_f1 = val_macro_f1 > best_val_macro_f1 + 1e-4
    improved_loss = val_loss < best_val_loss - 1e-4

    if improved_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch
        best_weights = model.get_weights()
        bad_epochs = 0
    else:
        bad_epochs += 1

    if improved_loss:
        best_val_loss = val_loss
        lr_bad_epochs = 0
    else:
        lr_bad_epochs += 1

    if lr_bad_epochs >= LR_PATIENCE:
        new_lr = max(float(optimizer.learning_rate.numpy()) * LR_FACTOR, MIN_LR)
        optimizer.learning_rate.assign(new_lr)
        lr_bad_epochs = 0
        print(f"  ReduceLROnPlateau manual -> lr={new_lr:.6f}")

    if bad_epochs >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch} | "
            f"best_epoch={best_epoch} best_val_macro_f1={best_val_macro_f1:.4f}"
        )
        break

# ===================== RESTORE BEST =====================
if best_weights is not None:
    model.set_weights(best_weights)

print("Best epoch:", best_epoch)
print("Best val_macro_f1:", round(best_val_macro_f1, 4))

Epoch 001 | train_loss=0.5983 train_acc=0.7095 | val_loss=0.5134 val_acc=0.8200 val_macro_f1=0.7511 | lr=0.001000
Epoch 002 | train_loss=0.3887 train_acc=0.8576 | val_loss=0.4055 val_acc=0.8711 val_macro_f1=0.8244 | lr=0.001000
Epoch 003 | train_loss=0.3191 train_acc=0.8895 | val_loss=0.3300 val_acc=0.9111 val_macro_f1=0.8748 | lr=0.001000
Epoch 004 | train_loss=0.2677 train_acc=0.9133 | val_loss=0.2790 val_acc=0.9244 val_macro_f1=0.8886 | lr=0.001000
Epoch 005 | train_loss=0.2990 train_acc=0.9090 | val_loss=0.3089 val_acc=0.8933 val_macro_f1=0.8338 | lr=0.001000
Epoch 006 | train_loss=0.3052 train_acc=0.9019 | val_loss=0.2576 val_acc=0.9222 val_macro_f1=0.8844 | lr=0.001000
Epoch 007 | train_loss=0.2464 train_acc=0.9381 | val_loss=0.2328 val_acc=0.9467 val_macro_f1=0.9238 | lr=0.001000
Epoch 008 | train_loss=0.2267 train_acc=0.9286 | val_loss=0.2240 val_acc=0.9400 val_macro_f1=0.9102 | lr=0.001000
Epoch 009 | train_loss=0.2149 train_acc=0.9524 | val_loss=0.2043 val_acc=0.9511 val_macr

## 14. Model Evaluation

Evaluasi final mencakup accuracy, macro/weighted F1, precision/recall/F1 per class, confusion matrix, classification report, dan error analysis singkat.

In [16]:
def predict_probs(ds: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    probs_all = []
    y_all = []
    for xb, yb in ds:
        probs = model(xb, training=False).numpy()
        probs_all.append(probs)
        y_all.append(np.argmax(yb.numpy(), axis=1))
    return np.concatenate(y_all, axis=0), np.concatenate(probs_all, axis=0)


def acc(y_true: np.ndarray, probs: np.ndarray) -> float:
    return float((np.argmax(probs, axis=1) == y_true).mean())


y_tr, p_tr = predict_probs(train_ds)
y_va, p_va = predict_probs(val_ds)
y_te, p_te = predict_probs(test_ds)

y_pred = np.argmax(p_te, axis=1)
conf = np.max(p_te, axis=1)

train_accuracy = acc(y_tr, p_tr)
val_accuracy = acc(y_va, p_va)
test_accuracy = acc(y_te, p_te)

macro_f1 = float(f1_score(y_te, y_pred, average="macro"))
weighted_f1 = float(f1_score(y_te, y_pred, average="weighted"))

report = classification_report(
    y_te,
    y_pred,
    labels=[0, 1, 2],
    target_names=[LABEL_MAPPING[str(i)] for i in [0, 1, 2]],
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_te, y_pred, labels=[0, 1, 2]).tolist()

per_class_precision = {str(k): float(report[LABEL_MAPPING[str(k)]]["precision"]) for k in [0, 1, 2]}
per_class_recall = {str(k): float(report[LABEL_MAPPING[str(k)]]["recall"]) for k in [0, 1, 2]}
per_class_f1 = {str(k): float(report[LABEL_MAPPING[str(k)]]["f1-score"]) for k in [0, 1, 2]}

print("Train accuracy:", round(train_accuracy, 4))
print("Validation accuracy:", round(val_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))
print("Per-class recall:", {LABEL_MAPPING[k]: round(v, 4) for k, v in per_class_recall.items()})
print("Confusion matrix [survival, stable, growth]:")
print(np.array(cm))

# Error analysis
errors = np.where(y_pred != y_te)[0]
print("\nTotal test errors:", len(errors), "out of", len(y_te))

pair_counts = {}
for idx in errors:
    pair = (int(y_te[idx]), int(y_pred[idx]))
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

if pair_counts:
    print("Most common confusion pairs (true->pred):")
    for (t, p), n in sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)[:10]:
        print(f"  {LABEL_MAPPING[str(t)]}->{LABEL_MAPPING[str(p)]}: {n}")

if len(errors) > 0:
    show_n = min(10, len(errors))
    sample_err = errors[:show_n]
    err_rows = X_test_df.iloc[sample_err].copy()
    err_rows["true_label"] = [LABEL_MAPPING[str(int(v))] for v in y_te[sample_err]]
    err_rows["pred_label"] = [LABEL_MAPPING[str(int(v))] for v in y_pred[sample_err]]
    err_rows["confidence"] = conf[sample_err]
    display(err_rows[["true_label", "pred_label", "confidence"]].head(show_n))

Train accuracy: 0.9719
Validation accuracy: 0.9644
Test accuracy: 0.9556
Macro F1: 0.9385
Weighted F1: 0.9569
Per-class recall: {'survival': 1.0, 'stable': 0.9394, 'growth': 0.9441}
Confusion matrix [survival, stable, growth]:
[[ 98   0   0]
 [  3  62   1]
 [  0  16 270]]

Total test errors: 20 out of 450
Most common confusion pairs (true->pred):
  growth->stable: 16
  stable->survival: 3
  stable->growth: 1


,true_label,pred_label,confidence
426,growth,stable,0.774113
1595,growth,stable,0.760757
2658,growth,stable,0.704269
705,growth,stable,0.793442
2583,growth,stable,0.693131
1486,stable,survival,0.602166
2845,growth,stable,0.934089
2052,stable,survival,0.501177
2134,growth,stable,0.508629
1255,stable,growth,0.520803


## 15. Save Final Artifacts

Simpan model, scaler, feature columns, label mapping, dan metrics final ke folder artifacts.

In [17]:
model_path = ARTIFACT_DIR / "classification_model.keras"
scaler_path = ARTIFACT_DIR / "classification_scaler.joblib"
feat_cols_path = ARTIFACT_DIR / "classification_feature_columns.json"
label_map_path = ARTIFACT_DIR / "classification_label_mapping.json"
metrics_path = ARTIFACT_DIR / "classification_metrics.json"
feature_selection_path = ARTIFACT_DIR / "classification_feature_selection_report.json"

model.save(model_path)
joblib.dump(scaler, scaler_path)

with open(feat_cols_path, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(LABEL_MAPPING, f, indent=2)

with open(feature_selection_path, "w", encoding="utf-8") as f:
    json.dump(feature_selection_report, f, indent=2)

metrics = {
    "model_name": model.name,
    "dataset_name": DATA_PATH.name,
    "target_column": TARGET,
    "label_mapping": LABEL_MAPPING,
    "feature_count": int(len(feature_columns)),
    "feature_columns": feature_columns,
    "feature_selection": feature_selection_report,
    "class_distribution": {
        "train": label_dist(y_train).to_dict(orient="index"),
        "val": label_dist(y_val).to_dict(orient="index"),
        "test": label_dist(y_test).to_dict(orient="index"),
    },
    "train_accuracy": float(train_accuracy),
    "val_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "per_class_precision": per_class_precision,
    "per_class_recall": per_class_recall,
    "per_class_f1": per_class_f1,
    "confusion_matrix": cm,
    "loss_function": "weighted_categorical_crossentropy(class_weight)",
    "optimizer": optimizer_name,
    "epochs_trained": int(len(history)),
    "best_epoch": int(best_epoch),
    "notes": "Custom GradientTape training loop + custom residual block + seleksi fitur otomatis.",
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved artifacts:")
print("-", model_path)
print("-", scaler_path)
print("-", feat_cols_path)
print("-", label_map_path)
print("-", feature_selection_path)
print("-", metrics_path)

Saved artifacts:
- artifacts\classification_model.keras
- artifacts\classification_scaler.joblib
- artifacts\classification_feature_columns.json
- artifacts\classification_label_mapping.json
- artifacts\classification_feature_selection_report.json
- artifacts\classification_metrics.json


## 16. Uji Inferensi Lokal (Input IDR)

Pada bagian ini kita meniru pola inference seperti di `finary_insight_model.ipynb`: input user dalam **IDR**, lalu dikonversi ke skala training dengan `UNIT_SCALE` sebelum masuk `scaler` dan model. Ini memastikan jalur inference konsisten dengan produksi.

In [18]:
import inspect

# Kontrak unit (selaras dengan insight + API): input user dalam IDR, lalu diskalakan ke unit training.
UNIT_SCALE = 2500.0

# ==========================================================
# Metode inference production-ready: input minimal dari user
# ==========================================================
# Wajib (IDR): monthly_income, monthly_expense_total, actual_savings, emergency_fund, budget_goal
# Opsional: credit_score, loan_payment, investment_amount, subscription_services, transaction_count,
#          rent_or_mortgage, discretionary_spending, essential_spending, main_category, fraud_flag

def build_classification_features_idr_minimal(payload: dict, *, unit_scale: float = UNIT_SCALE) -> dict:
    required_keys = [
        "monthly_income",
        "monthly_expense_total",
        "actual_savings",
        "emergency_fund",
        "budget_goal",
    ]
    missing = [k for k in required_keys if payload.get(k, None) is None]
    if missing:
        raise ValueError(f"Field wajib belum ada: {missing}")

    def f(key: str, default: float = 0.0) -> float:
        v = payload.get(key, default)
        return float(v) if v is not None else float(default)

    # --- Uang: IDR -> unit training ---
    inc = f("monthly_income") / unit_scale
    exp = f("monthly_expense_total") / unit_scale
    savings = f("actual_savings") / unit_scale
    emergency_fund = f("emergency_fund") / unit_scale
    budget_goal = f("budget_goal") / unit_scale

    loan_payment = f("loan_payment", 0.0) / unit_scale
    investment_amount = f("investment_amount", 0.0) / unit_scale
    rent = f("rent_or_mortgage", 0.0) / unit_scale

    # Jika rincian spending tidak diinput, aproksimasi dari total expense.
    discretionary_idr = payload.get("discretionary_spending", None)
    essential_idr = payload.get("essential_spending", None)
    if discretionary_idr is None or essential_idr is None:
        discretionary = 0.30 * exp
        essential = 0.70 * exp
    else:
        discretionary = float(discretionary_idr) / unit_scale
        essential = float(essential_idr) / unit_scale

    # --- Non-uang + default ---
    credit_score = f("credit_score", 650.0)
    subscription_services = f("subscription_services", 0.0)
    transaction_count = f("transaction_count", 0.0)
    fraud_flag = f("fraud_flag", 0.0)

    # DTI: turunkan dari cicilan/income bila tidak ada.
    dti = f("debt_to_income_ratio", (loan_payment / inc if inc > 0 else 0.0))

    # --- Engineered features ---
    net_cash_flow = inc - exp
    expense_ratio = exp / inc if inc > 0 else 0.0
    savings_rate = savings / inc if inc > 0 else 0.0
    financial_buffer = emergency_fund / exp if exp > 0 else 0.0

    savings_goal_met = 1.0 if savings >= budget_goal else 0.0

    debt_pressure = loan_payment * dti
    spending_efficiency = (essential / exp) if exp > 0 else 0.0
    lifestyle_burden = (discretionary / inc) if inc > 0 else 0.0
    saving_behavior = (savings / budget_goal) if budget_goal > 0 else 0.0

    credit_norm = float(np.clip((credit_score - 300.0) / 550.0, 0.0, 1.0))
    buffer_norm = float(np.clip(financial_buffer / 3.0, 0.0, 1.0))
    advice_score = float(
        np.clip(
            (
                0.35 * credit_norm
                + 0.35 * buffer_norm
                + 0.15 * spending_efficiency
                + 0.15 * float(np.clip(savings_rate / 0.5, 0.0, 1.0))
            )
            * 100.0,
            0.0,
            100.0,
        )
    )

    stress_medium = 1.0 if (expense_ratio > 0.9 or dti >= 0.35) else 0.0

    # One-hot category (hanya yang ada di 27 fitur terpilih)
    main_category = str(payload.get("main_category", "")).strip().title()
    cat_education = 1.0 if main_category == "Education" else 0.0
    cat_entertainment = 1.0 if main_category == "Entertainment" else 0.0
    cat_transportation = 1.0 if main_category == "Transportation" else 0.0

    # Kunci: output sesuai schema 27 fitur
    features = {col: 0.0 for col in loaded_feature_columns}
    features.update(
        {
            "saving_behavior": saving_behavior,
            "expense_ratio": expense_ratio,
            "actual_savings": savings,
            "net_cash_flow": net_cash_flow,
            "monthly_income": inc,
            "monthly_expense_total": exp,
            "lifestyle_burden": lifestyle_burden,
            "savings_goal_met": savings_goal_met,
            "spending_efficiency": spending_efficiency,
            "financial_buffer": financial_buffer,
            "discretionary_spending": discretionary,
            "budget_goal": budget_goal,
            "savings_rate": savings_rate,
            "rent_or_mortgage": rent,
            "debt_pressure": debt_pressure,
            "emergency_fund": emergency_fund,
            "category_Education": cat_education,
            "financial_advice_score": advice_score,
            "credit_score": credit_score,
            "category_Entertainment": cat_entertainment,
            "investment_amount": investment_amount,
            "loan_payment": loan_payment,
            "subscription_services": subscription_services,
            "category_Transportation": cat_transportation,
            "financial_stress_level_Medium": stress_medium,
            "fraud_flag": fraud_flag,
            "transaction_count": transaction_count,
        }
    )

    return features


def _ensure_residual_dense_block_keras3_compatible():
    cls = globals().get("ResidualDenseBlock", None)
    needs_redefine = cls is None

    if cls is not None:
        sig = inspect.signature(cls.__init__)
        has_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
        needs_redefine = not has_kwargs

    if needs_redefine:
        @tf.keras.utils.register_keras_serializable(package="finary")
        class ResidualDenseBlock(tf.keras.layers.Layer):
            def __init__(
                self,
                units: int,
                dropout: float,
                l2: float,
                activation: str = "gelu",
                name: str | None = None,
                **kwargs,
            ):
                super().__init__(name=name, **kwargs)
                reg = tf.keras.regularizers.l2(l2)
                self.units = units
                self.dropout = dropout
                self.l2 = l2
                self.activation = activation

                self.dense1 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
                self.bn1 = tf.keras.layers.BatchNormalization()
                self.drop1 = tf.keras.layers.Dropout(dropout)

                self.dense2 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
                self.bn2 = tf.keras.layers.BatchNormalization()
                self.drop2 = tf.keras.layers.Dropout(dropout)

                self.proj = None

            def get_config(self):
                config = super().get_config()
                config.update(
                    {
                        "units": self.units,
                        "dropout": self.dropout,
                        "l2": self.l2,
                        "activation": self.activation,
                    }
                )
                return config

            def _act(self, x):
                return tf.keras.activations.gelu(x) if self.activation.lower() == "gelu" else tf.nn.relu(x)

            def build(self, input_shape):
                in_units = int(input_shape[-1])
                if in_units != self.units:
                    self.proj = tf.keras.layers.Dense(self.units)
                super().build(input_shape)

            def call(self, x, training=False):
                skip = self.proj(x) if self.proj is not None else x

                y = self.dense1(x)
                y = self.bn1(y, training=training)
                y = self._act(y)
                y = self.drop1(y, training=training)

                y = self.dense2(y)
                y = self.bn2(y, training=training)
                y = self._act(y)
                y = self.drop2(y, training=training)

                return skip + y

        globals()["ResidualDenseBlock"] = ResidualDenseBlock


def build_classification_features_idr(payload: dict) -> dict:
    """Bangun fitur model klasifikasi dari payload IDR.

    Tujuan fungsi ini: meniru pola API/insight: IDR -> unit training -> fitur turunan.
    Semua fitur yang tidak ada di schema model akan diisi 0.
    """

    def _f(key: str, default: float = 0.0) -> float:
        v = payload.get(key, default)
        return float(v) if v is not None else float(default)

    # Uang (IDR -> unit training)
    inc = _f("monthly_income") / UNIT_SCALE
    exp = _f("monthly_expense_total") / UNIT_SCALE
    budget_goal = _f("budget_goal") / UNIT_SCALE
    loan_payment = _f("loan_payment") / UNIT_SCALE
    investment_amount = _f("investment_amount") / UNIT_SCALE
    emergency_fund = _f("emergency_fund") / UNIT_SCALE
    discretionary = _f("discretionary_spending") / UNIT_SCALE
    essential = _f("essential_spending") / UNIT_SCALE
    rent = _f("rent_or_mortgage") / UNIT_SCALE
    savings = _f("actual_savings") / UNIT_SCALE

    # Non-uang
    credit_score = _f("credit_score")
    dti = _f("debt_to_income_ratio")
    subscription_services = _f("subscription_services")
    transaction_count = _f("transaction_count")

    # Fitur turunan (selaras dengan `api_service.py`)
    net_cash_flow = inc - exp
    expense_ratio = exp / inc if inc > 0 else 0.0
    savings_rate = savings / inc if inc > 0 else 0.0
    financial_buffer = emergency_fund / exp if exp > 0 else 0.0

    savings_goal_met = 1.0 if savings >= budget_goal else 0.0
    debt_ratio_flag = 1.0 if dti >= 0.35 else 0.0
    low_emergency_flag = 1.0 if financial_buffer < 1.0 else 0.0

    debt_pressure = loan_payment * dti
    spending_efficiency = (essential / exp) if exp > 0 else 0.0
    lifestyle_burden = (discretionary / inc) if inc > 0 else 0.0
    saving_behavior = (savings / budget_goal) if budget_goal > 0 else 0.0

    # Skor advice proxy (menghindari feature 0 terus bila dipilih feature selection)
    credit_norm = float(np.clip((credit_score - 300.0) / 550.0, 0.0, 1.0))
    buffer_norm = float(np.clip(financial_buffer / 3.0, 0.0, 1.0))
    advice_score = float(
        np.clip(
            (
                0.35 * credit_norm
                + 0.35 * buffer_norm
                + 0.15 * spending_efficiency
                + 0.15 * float(np.clip(savings_rate / 0.5, 0.0, 1.0))
            )
            * 100.0,
            0.0,
            100.0,
        )
    )

    income_type = str(payload.get("income_type", "Salary"))
    main_category = str(payload.get("main_category", "Utilities"))

    features = {col: 0.0 for col in loaded_feature_columns}
    features.update(
        {
            "monthly_income": inc,
            "monthly_expense_total": exp,
            "savings_rate": savings_rate,
            "budget_goal": budget_goal,
            "credit_score": credit_score,
            "debt_to_income_ratio": dti,
            "loan_payment": loan_payment,
            "investment_amount": investment_amount,
            "subscription_services": subscription_services,
            "emergency_fund": emergency_fund,
            "transaction_count": transaction_count,
            "discretionary_spending": discretionary,
            "essential_spending": essential,
            "rent_or_mortgage": rent,
            "financial_advice_score": advice_score,
            "actual_savings": savings,
            "savings_goal_met": savings_goal_met,
            "net_cash_flow": net_cash_flow,
            "expense_ratio": expense_ratio,
            "debt_pressure": debt_pressure,
            "financial_buffer": financial_buffer,
            "saving_behavior": saving_behavior,
            "spending_efficiency": spending_efficiency,
            "lifestyle_burden": lifestyle_burden,
            "debt_ratio_flag": debt_ratio_flag,
            "low_emergency_flag": low_emergency_flag,
        }
    )

    inc_type_col = f"income_type_{income_type}"
    if inc_type_col in features:
        features[inc_type_col] = 1.0

    category_col = f"category_{main_category}"
    if category_col in features:
        features[category_col] = 1.0

    if "cash_flow_status_Positive" in features:
        features["cash_flow_status_Positive"] = 1.0 if net_cash_flow > 0 else 0.0
    if "cash_flow_status_Neutral" in features:
        features["cash_flow_status_Neutral"] = 1.0 if net_cash_flow <= 0 else 0.0

    if "financial_stress_level_Medium" in features:
        features["financial_stress_level_Medium"] = 1.0 if (expense_ratio > 0.9 or dti >= 0.35) else 0.0
    if "financial_stress_level_Low" in features:
        features["financial_stress_level_Low"] = 0.0 if features.get("financial_stress_level_Medium", 0.0) == 1.0 else 1.0

    return features


_ensure_residual_dense_block_keras3_compatible()

loaded_model = tf.keras.models.load_model(
    ARTIFACT_DIR / "classification_model.keras",
    custom_objects={"ResidualDenseBlock": ResidualDenseBlock},
    compile=False,
)
loaded_scaler = joblib.load(ARTIFACT_DIR / "classification_scaler.joblib")
with open(ARTIFACT_DIR / "classification_feature_columns.json", "r", encoding="utf-8") as f:
    loaded_feature_columns = json.load(f)
with open(ARTIFACT_DIR / "classification_label_mapping.json", "r", encoding="utf-8") as f:
    loaded_label_mapping = json.load(f)

# Contoh payload inference (IDR) - format mendekati API
sample_payload_idr = {
    "monthly_income": 10_000_000,
    "monthly_expense_total": 6_000_000,
    "budget_goal": 5_000_000,
    "credit_score": 700,
    "debt_to_income_ratio": 0.25,
    "loan_payment": 0,
    "investment_amount": 1_000_000,
    "subscription_services": 3,
    "emergency_fund": 1_000_000,
    "transaction_count": 40,
    "discretionary_spending": 1_500_000,
    "essential_spending": 4_500_000,
    "rent_or_mortgage": 0,
    "actual_savings": 1_000_000,
    "income_type": "Salary",
    "main_category": "Utilities",
}

features = build_classification_features_idr_minimal(sample_payload_idr)
df_input = pd.DataFrame([features]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
X_scaled = loaded_scaler.transform(df_input.values)

probs = loaded_model.predict(X_scaled, verbose=0)[0]
pred_id = int(np.argmax(probs))

inference_output = {
    "prediction": loaded_label_mapping[str(pred_id)],
    "confidence": float(np.max(probs)),
    "probabilities": {loaded_label_mapping[str(i)]: float(probs[i]) for i in range(3)},
}
inference_output

{'prediction': 'growth',
 'confidence': 0.9935897588729858,
 'probabilities': {'survival': 9.14693737286143e-05,
  'stable': 0.006318757776170969,
  'growth': 0.9935897588729858}}

In [19]:
# Uji inferensi berbasis skenario (input IDR) - konsisten dengan kontrak unit

scenario_payloads_idr = {
    "survival": {
        "monthly_income": 7_000_000,
        "monthly_expense_total": 8_000_000,
        "budget_goal": 2_000_000,
        "credit_score": 620,
        "debt_to_income_ratio": 0.55,
        "loan_payment": 1_500_000,
        "investment_amount": 0,
        "subscription_services": 5,
        "emergency_fund": 500_000,
        "transaction_count": 70,
        "discretionary_spending": 2_000_000,
        "essential_spending": 6_000_000,
        "rent_or_mortgage": 2_500_000,
        "actual_savings": 200_000,
        "income_type": "Salary",
        "main_category": "Utilities",
    },
    "stable": {
        "monthly_income": 12_000_000,
        "monthly_expense_total": 9_000_000,
        "budget_goal": 4_000_000,
        "credit_score": 700,
        "debt_to_income_ratio": 0.28,
        "loan_payment": 800_000,
        "investment_amount": 750_000,
        "subscription_services": 3,
        "emergency_fund": 15_000_000,
        "transaction_count": 55,
        "discretionary_spending": 2_500_000,
        "essential_spending": 6_500_000,
        "rent_or_mortgage": 2_500_000,
        "actual_savings": 1_500_000,
        "income_type": "Salary",
        "main_category": "Utilities",
    },
    "growth": {
        "monthly_income": 25_000_000,
        "monthly_expense_total": 12_000_000,
        "budget_goal": 8_000_000,
        "credit_score": 760,
        "debt_to_income_ratio": 0.12,
        "loan_payment": 0,
        "investment_amount": 4_000_000,
        "subscription_services": 2,
        "emergency_fund": 60_000_000,
        "transaction_count": 45,
        "discretionary_spending": 3_000_000,
        "essential_spending": 9_000_000,
        "rent_or_mortgage": 3_500_000,
        "actual_savings": 7_000_000,
        "income_type": "Mixed",
        "main_category": "Investments",
    },
}

rows = []
for name, payload in scenario_payloads_idr.items():
    feats = build_classification_features_idr_minimal(payload)
    df_input = pd.DataFrame([feats]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
    X_scaled = loaded_scaler.transform(df_input.values)
    p = loaded_model.predict(X_scaled, verbose=0)[0]
    pred_id = int(np.argmax(p))

    rows.append(
        {
            "skenario": name,
            "prediksi": loaded_label_mapping[str(pred_id)],
            "confidence": float(np.max(p)),
            "p_survival": float(p[0]),
            "p_stable": float(p[1]),
            "p_growth": float(p[2]),
        }
    )

hasil_df = pd.DataFrame(rows).sort_values(by="confidence", ascending=False)
print("Hasil uji skenario (input IDR):")
display(hasil_df)

rows

Hasil uji skenario (input IDR):


,skenario,prediksi,confidence,p_survival,p_stable,p_growth
2,growth,growth,0.999989,0.000010,3.909979e-07,0.999989
0,survival,survival,0.983205,0.983205,1.673295e-02,0.000062
1,stable,stable,0.522799,0.019155,5.227987e-01,0.458047


[{'skenario': 'survival',
  'prediksi': 'survival',
  'confidence': 0.9832046627998352,
  'p_survival': 0.9832046627998352,
  'p_stable': 0.016732946038246155,
  'p_growth': 6.232842861209065e-05},
 {'skenario': 'stable',
  'prediksi': 'stable',
  'confidence': 0.5227986574172974,
  'p_survival': 0.01915452629327774,
  'p_stable': 0.5227986574172974,
  'p_growth': 0.4580467641353607},
 {'skenario': 'growth',
  'prediksi': 'growth',
  'confidence': 0.9999892711639404,
  'p_survival': 1.0351902346883435e-05,
  'p_stable': 3.909979113814188e-07,
  'p_growth': 0.9999892711639404}]

In [21]:
# Uji inference dengan input MINIMAL (sesuai kontrak produksi)
# Hanya mengirim field wajib: monthly_income, monthly_expense_total, actual_savings, emergency_fund, budget_goal.

minimal_payloads_idr = {
    "survival": {
        "monthly_income": 7_000_000,
        "monthly_expense_total": 8_500_000,
        "actual_savings": 100_000,
        "emergency_fund": 300_000,
        "budget_goal": 2_000_000,
    },
    "stable": {
        "monthly_income": 11_000_000,
        "monthly_expense_total": 9_500_000,
        "actual_savings": 1_000_000,
        "emergency_fund": 8_000_000,
        "budget_goal": 3_500_000,
    },
    "growth": {
        "monthly_income": 25_000_000,
        "monthly_expense_total": 12_500_000,
        "actual_savings": 6_000_000,
        "emergency_fund": 60_000_000,
        "budget_goal": 8_000_000,
    },
}

rows = []
for name, payload in minimal_payloads_idr.items():
    feats = build_classification_features_idr_minimal(payload)
    df_input = pd.DataFrame([feats]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
    X_scaled = loaded_scaler.transform(df_input.values)
    p = loaded_model.predict(X_scaled, verbose=0)[0]
    pred_id = int(np.argmax(p))

    rows.append(
        {
            "skenario_input_minimal": name,
            "prediksi": loaded_label_mapping[str(pred_id)],
            "confidence": float(np.max(p)),
            "p_survival": float(p[0]),
            "p_stable": float(p[1]),
            "p_growth": float(p[2]),
        }
    )

hasil_minimal_df = pd.DataFrame(rows).sort_values(by="confidence", ascending=False)
print("Hasil uji skenario (input minimal wajib saja, IDR):")
display(hasil_minimal_df)

rows

Hasil uji skenario (input minimal wajib saja, IDR):


,skenario_input_minimal,prediksi,confidence,p_survival,p_stable,p_growth
2,growth,growth,0.999958,0.000040,0.000002,0.999958
0,survival,survival,0.994900,0.994900,0.005098,0.000002
1,stable,stable,0.914466,0.006693,0.914466,0.078842


[{'skenario_input_minimal': 'survival',
  'prediksi': 'survival',
  'confidence': 0.9949000477790833,
  'p_survival': 0.9949000477790833,
  'p_stable': 0.005097663961350918,
  'p_growth': 2.29316856348305e-06},
 {'skenario_input_minimal': 'stable',
  'prediksi': 'stable',
  'confidence': 0.914465606212616,
  'p_survival': 0.006692657247185707,
  'p_stable': 0.914465606212616,
  'p_growth': 0.0788416787981987},
 {'skenario_input_minimal': 'growth',
  'prediksi': 'growth',
  'confidence': 0.9999581575393677,
  'p_survival': 3.9714181184535846e-05,
  'p_stable': 2.2036438167560846e-06,
  'p_growth': 0.9999581575393677}]

## 17. Ringkasan Akhir

Model ini memprediksi kondisi keuangan bulanan personal dengan target baru `survival/stable/growth`. Target ini selaras dengan arus kas (cashflow) dan sangat berkaitan dengan indikator utama seperti `net_cash_flow`, `expense_ratio`, dan `actual_savings`. Arsitektur yang digunakan adalah TensorFlow Residual MLP (Functional API) dengan komponen kustom `ResidualDenseBlock` serta training loop berbasis `tf.GradientTape`, lalu diekspor menjadi artifact yang siap dipakai untuk inference di produksi.